# PerfectPetClub - Data Engineering ETL Pipeline
This notebook  Extract, Transform and Loads supplied database.

### The pipeline:
Extracts records from a CSV file.

Cleans and standardises extracted data.

Exports the cleaned dataset in CSV and Parquet formats.

Displays the original and transformed datasets for validation.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))


%load_ext autoreload
%autoreload 2


In [ ]:

from crm import run_crm_etl

# Call the function to run  crm etl file 
crm_df, cleaned_crm_df,crm_features_df = run_crm_etl()

# Display the first five rows to confirm to check extraction and transformation 
display(crm_df.head())

display(cleaned_crm_df.head())

display(crm_features_df.head())

In [ ]:
from extract_customer_subscriptions import (
    extract_customer_subscriptions,
    validate_customer_data,
    transform_customer_features
)


print('Environment and modules loaded successfully!')

In [ ]:
DB_PATH = '../data/PerfectPetClub.db'

raw_customers = extract_customer_subscriptions(db_path=DB_PATH)

print(f'Extracted {len(raw_customers)} customer records.')
raw_customers.head(3)

In [ ]:
print("--- Validating Customer Data ---")

cust_dq = validate_customer_data(raw_customers)

print("Customer validation results:")
for check, result in cust_dq.items():
    print(f"{check}: {result}")

In [ ]:

 
# Test: Can our validation detect an invalid date?

test_data = raw_customers.copy()

# Deliberately insert an invalid subscription date
test_data.loc[0, 'sub_end_date'] = 'invalid-date'

try:
    validate_customer_data(test_data)
    print("TEST FAILED: Invalid date was accepted.")

except AssertionError as e:
    print("TEST PASSED: Invalid date was detected.")
    print(e)



 

In [ ]:
# Transform customer and subscription data
clean_customers = transform_customer_features(raw_customers)

# Prepare customer-only dataset
final_dataset = clean_customers.drop(
    columns=['customer_name', 'postcode', 'subscription_id']
)

print("Total customers:", len(final_dataset))
print("Dataset shape:", final_dataset.shape)

display(final_dataset.head())

In [ ]:
# Analyse churn rate by subscription tier

tier_summary = final_dataset.groupby("subscription_tier")["is_churned"].agg(
    total_customers="count",
    churn_rate="mean"
).reset_index()

# Convert churn rate into percentage
tier_summary["churn_rate"] = (
    tier_summary["churn_rate"] * 100
).round(2)

# Display results
print("--- Churn Rate by Subscription Tier ---")
display(tier_summary)

In [ ]:
# Create output directory
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / 'perfectpetclub_churn_features.parquet'

# Export to Parquet
final_dataset.to_parquet(parquet_path, index=False, engine='pyarrow')

print(f'✓ Successfully loaded {len(final_dataset)} records into Parquet format:')
print(f'  Location: {parquet_path.resolve()}')
print(f'  File size: {parquet_path.stat().st_size:,} bytes')


In [ ]:
# Verification: Read back parquet file to guarantee integrity
df_verified = pd.read_parquet(parquet_path)
print(f'Parquet verification passed: {df_verified.shape[0]} rows, {df_verified.shape[1]} columns.')
print('\nDataset Columns:')
for col in df_verified.columns:
    print(f' - {col} ({df_verified[col].dtype})')

In [ ]:
# Final verification of feature engineering and Parquet output

# Check 1: Customer records have not been duplicated
assert final_dataset['customer_id'].is_unique, "Duplicate customers found"

# Check 2: Customer count remains unchanged
assert len(final_dataset) == len(raw_customers), "Customer count mismatch"

# Check 3: Churn calculation is correct
expected_churn = (final_dataset['auto_renew'] == 0).astype(int)
assert (final_dataset['is_churned'] == expected_churn).all(), "Incorrect churn values"

# Check 4: Subscription durations are not negative
assert (final_dataset['subscription_tenure_days'] >= 0).all(), "Negative duration found"

# Check 5: Parquet file contains the expected records and columns
assert df_verified.shape == final_dataset.shape, "Parquet dataset shape mismatch"
assert list(df_verified.columns) == list(final_dataset.columns), "Parquet columns mismatch"

print("PASS: Customer records are unique")
print("PASS: Customer count is correct")
print("PASS: Churn calculations are correct")
print("PASS: Subscription durations are valid")
print("PASS: Parquet structure is verified")

In [49]:
from pathlib import Path
import importlib.util
import pandas as pd

# Find the project folder containing src/ and data/
current_dir = Path.cwd().resolve()
project_root = next(
    (folder for folder in [current_dir, *current_dir.parents]
     if (folder / 'src' / 'server.py').is_file()
     and (folder / 'data').is_dir()),
    None
)

if project_root is None:
    raise FileNotFoundError('Could not find the project folder with src/ and data/.')

# Import functions from the project's server.py
server_file = project_root / 'src' / 'server.py'
spec = importlib.util.spec_from_file_location('perfectpetclub_etl_server', server_file)
server_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(server_module)

extract_data = server_module.extract_data
transform_data = server_module.transform_data
load_data = server_module.load_data

print('ETL functions imported successfully.')

ETL functions imported successfully.


In [50]:
input_file = project_root / 'data' / 'PerfectPetClub_server_log.csv'
output_dir = project_root / 'output'
parquet_file = output_dir / 'perfectpetclub_customer_features.parquet'

if not input_file.is_file():
    raise FileNotFoundError(f'Input CSV not found: {input_file}')

output_dir.mkdir(parents=True, exist_ok=True)
print('Input file:', input_file)
print('Output folder:', output_dir)

Input file: C:\Users\bl26aan\OneDrive - University of Hertfordshire\7COM2006\Data Engineering Team Presentation\DataEngineeringPresentation\data\PerfectPetClub_server_log.csv
Output folder: C:\Users\bl26aan\OneDrive - University of Hertfordshire\7COM2006\Data Engineering Team Presentation\DataEngineeringPresentation\output


In [58]:
def run_pipeline(input_path, output_dir):
    # EXTRACT
    print('STEP 1 - EXTRACT')
    raw_data = extract_data(input_path)
    print('Raw data shape:', raw_data.shape)

    # TRANSFORM
    print('\nSTEP 2 - TRANSFORM')
    clean_data, feature_data, quality_report = transform_data(raw_data)
    print('Cleaned event data:', clean_data.shape)
    print('Customer feature data:', feature_data.shape)

    # LOAD - save the final dataset as a Parquet file
    print('\nSTEP 3 - LOAD')
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save final customer-level dataset
    parquet_path = output_dir / 'perfectpetclub_final_dataset.parquet'
    
    parquet_path = load_data(feature_data, parquet_path)
    
    print('Saved final dataset as Parquet:', parquet_path)

    # VALIDATE saved customer-level Parquet
    print('\nSTEP 4 - VALIDATE')
    final_data = pd.read_parquet(parquet_path)
    assert final_data['customer_id'].is_unique
    assert len(final_data) == len(feature_data)
    assert 'churn_label' in final_data.columns
    print('Saved data validation passed.')
    print('Final customer records:', len(final_data))

    return final_data, clean_data, feature_data, quality_report

final_data, clean_data, feature_data, quality_report = run_pipeline(input_file, output_dir)

STEP 1 - EXTRACT
Extracted 67,249 rows from PerfectPetClub_server_log.csv
Raw data shape: (67249, 5)

STEP 2 - TRANSFORM
Transformed 67,249 clean events into 221 customer records
Cleaned event data: (67249, 12)
Customer feature data: (221, 21)

STEP 3 - LOAD
Saved 221 customer records to C:\Users\bl26aan\OneDrive - University of Hertfordshire\7COM2006\Data Engineering Team Presentation\DataEngineeringPresentation\output\perfectpetclub_final_dataset.parquet
Saved final dataset as Parquet: C:\Users\bl26aan\OneDrive - University of Hertfordshire\7COM2006\Data Engineering Team Presentation\DataEngineeringPresentation\output\perfectpetclub_final_dataset.parquet

STEP 4 - VALIDATE
Saved data validation passed.
Final customer records: 221


In [62]:
print('CLEANED EVENT DATA')
display(clean_data.head(10))

print('CUSTOMER-LEVEL FEATURE DATA')
display(feature_data.head(10))

print('ETL QUALITY REPORT')
quality_report_df = pd.DataFrame(
    list(quality_report.items()),
    columns=['quality_check', 'result']
)
display(quality_report_df)